# Calculate ROAS by platform and campaign

*Data Analysis in Python · Lesson 5 final exercise*

The media team is splitting next year's budget, and wants ROAS by platform and by campaign for 2024 and 2025: the store's revenue for every dollar each platform spent.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

Add your cells between the cell that reads the three spend files and the **Check your work** cell. If you're stuck on a step, open its hint under the **Check your work** cell.

1. Create the `spend` table: the three platforms' spend for 2024 and 2025 in the standard columns in `COLUMNS`, with `platform` as `"Google Ads"`, `"Meta"` or `"YouTube"`, and `spend` in dollars. Check each platform's total against its file.
2. Create the `revenue` table: one row per date and campaign of the `sessions` table, with the columns `date`, `campaign`, `sessions`, `transactions` and `revenue`.
3. Make the campaign names of the `spend` and `revenue` tables match, with one function on both, and check that every campaign in the `spend` table is in the `revenue` table.
4. Create the `campaign_days` table: one row per date, platform and campaign with spend, with the columns `date`, `platform`, `campaign`, `impressions`, `clicks` and `spend`, joined to the `revenue` table on the date and the campaign. Make the merge stop if a date and campaign appear more than once on either side, and check that every row found its revenue. Days after a campaign's last spend, with only refunds, aren't in the table.
5. Create the `roas_by_platform` table: one row per platform, with the platforms as the index, and the columns `spend`, `revenue` and `roas`.
6. Create the `roas_by_campaign` table: one row per campaign, with the columns `platform`, `campaign`, `spend`, `revenue` and `roas`, sorted by `roas`, highest first.
7. Add a text cell for the media team: which platform returns the most for each dollar, and which campaign would you look at first, and why? Give the numbers.
8. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `spend` table has 21,536 rows and $571,200.03 of spend. The `campaign_days` table has 13,862 rows, one per date and campaign, with revenue on every row. The `roas_by_platform` table is Google Ads 2.28, Meta 2.02 and YouTube 1.62. The `roas_by_campaign` table has 84 campaigns, sorted by ROAS, from `2024_ca_display_blackfriday` (9.59) to `2025_us_facebook_blackfriday` (1.00).

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = [f"{year}-{month:02d}" for year in (2024, 2025) for month in range(1, 13)]
START, END = "2024-01-01", "2025-12-31"
COLUMNS = ["date", "platform", "campaign", "ad_set", "impressions", "clicks", "spend"]

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# The three platforms' exports, as they come. Run it, and leave it as it is.
google = pd.read_csv(DATA_URL + "google_ads_spend.csv", parse_dates=["date"])
meta = pd.read_csv(DATA_URL + "meta_ads_spend.csv", parse_dates=["Reporting starts", "Reporting ends"])
youtube = pd.read_csv(DATA_URL + "youtube_spend.csv", parse_dates=["Day"])
google.shape, meta.shape, youtube.shape

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    platforms = {"Google Ads": (357599.94, 2.2766), "Meta": (141600.04, 2.0194), "YouTube": (72000.05, 1.6192)}
    spend = found.get("spend")
    revenue = found.get("revenue")
    if not isinstance(spend, pd.DataFrame):
        results.append((False, "spend isn't a DataFrame yet. Map each platform to COLUMNS with rename(), add platform and spend, and stack the three with pd.concat (topic 5.1)."))
    elif list(spend.columns) != ["date", "platform", "campaign", "ad_set", "impressions", "clicks", "spend"]:
        results.append((False, f"The spend table's columns should be COLUMNS, in that order. They are {list(spend.columns)}."))
    elif len(spend) != 21536:
        results.append((False, f"The spend table has {len(spend):,} rows, but the three files have 21,536: 7,602 from Google Ads, 12,424 from Meta and 1,510 from YouTube. Keep every row of both years."))
    elif sorted(spend["platform"].unique()) != sorted(platforms):
        results.append((False, f"The spend table's platform column should be Google Ads, Meta or YouTube. It has {sorted(spend['platform'].unique())}."))
    elif spend.loc[spend["platform"] == "Google Ads", "spend"].sum() > 1_000_000:
        results.append((False, "Google Ads' spend in the spend table is in the billions: cost_micros wasn't divided. spend is cost_micros / 1_000_000 (topic 4.1)."))
    elif any(abs(spend.loc[spend["platform"] == platform, "spend"].sum() - total) > 0.01 for platform, (total, roas) in platforms.items()):
        totals = spend.groupby("platform")["spend"].sum()
        results.append((False, f"The spend table's platform totals should be Google Ads $357,599.94, Meta $141,600.04 and YouTube $72,000.05. They are {', '.join(f'{platform} ${total:,.2f}' for platform, total in totals.items())}."))
    elif not isinstance(revenue, pd.DataFrame) or not {"date", "campaign", "revenue"} <= set(revenue.columns) or len(revenue) != 18956:
        results.append((False, "revenue should be one row per date and campaign of the sessions table, 18,956 rows, with the columns date, campaign, sessions, transactions and revenue (topic 5.2)."))
    elif unmatched := sorted(set(spend["campaign"]) - set(revenue["campaign"])):
        results.append((False, f"{len(unmatched)} of the spend table's campaign names aren't in the revenue table, such as {unmatched[:3]}. Strip and lowercase both tables' names with one function (topic 5.3)."))
    else:
        results.append((True, "The spend table has both years of all three platforms, $571,200.03, and every campaign name matches the revenue table."))

    days = found.get("campaign_days")
    if not isinstance(days, pd.DataFrame):
        results.append((False, "campaign_days isn't a DataFrame yet. Group the spend table by date, platform and campaign, and join it to the revenue table with how=\"left\"."))
    elif "ad_set" in days.columns or len(days) == 21536:
        results.append((False, f"The campaign_days table has {len(days):,} rows: one per Meta ad set, so every ad set got its campaign-day's revenue. Group the spend table by date, platform and campaign before you join (topic 5.4)."))
    elif len(days) > 13862:
        results.append((False, f"The campaign_days table has {len(days):,} rows, but there are 13,862 campaign-days with spend. Keep the spend side's rows only: how=\"left\"."))
    elif len(days) != 13862 or not {"date", "platform", "campaign", "spend", "revenue"} <= set(days.columns):
        results.append((False, f"The campaign_days table should have 13,862 rows, one per date and campaign with spend, and the columns date, platform, campaign, spend and revenue. It has {len(days):,} rows."))
    elif days["revenue"].isna().any():
        results.append((False, f"{days['revenue'].isna().sum():,} rows of the campaign_days table have no revenue. Fix the campaign names before you join."))
    elif abs(days["spend"].sum() - 571200.03) > 0.01 or abs(days["revenue"].sum() - 1216642.52) > 0.01:
        results.append((False, f"The campaign_days table's spend and revenue add up to ${days['spend'].sum():,.2f} and ${days['revenue'].sum():,.2f}, but they should be $571,200.03 and $1,216,642.52."))
    elif history is not None and not any("validate=\"one_to_one\"" in source.replace("'", "\"").replace(" ", "") or "validate=\"1:1\"" in source.replace("'", "\"").replace(" ", "") for source in history):
        results.append((False, "The campaign_days table is right, but no merge in your notebook says validate=\"one_to_one\". Add it, so next year's join stops if a key repeats."))
    else:
        results.append((True, "The campaign_days table has 13,862 campaign-days, each with its spend and its revenue once."))

    by_platform = found.get("roas_by_platform")
    if not isinstance(by_platform, pd.DataFrame) or "roas" not in by_platform.columns:
        results.append((False, "roas_by_platform should be a DataFrame with the platforms as its index and the columns spend, revenue and roas."))
    elif sorted(by_platform.index) != sorted(platforms):
        results.append((False, f"The roas_by_platform table's index should be the three platforms. It is {list(by_platform.index)}. Group by platform, without as_index=False."))
    else:
        roas = by_platform["roas"]
        if abs(roas["Meta"] - 5.2234) < 0.0001 or abs(roas["Meta"] - 3.9135) < 0.0001:
            results.append((False, f"Meta's ROAS is {roas['Meta']:.2f}: the naive merge, with each campaign-day's revenue copied onto every ad set. Aggregate to campaign × day first."))
        elif abs(roas["YouTube"]) < 0.0001:
            results.append((False, "YouTube's ROAS is 0: its campaign names have capital markets, so none of them matched the revenue table. Fix the keys on both sides."))
        elif abs(roas["Meta"] - 1.5567) < 0.0001:
            results.append((False, "Meta's ROAS is 1.56: the campaigns whose names end in a space found no revenue. Strip the names."))
        elif roas["Google Ads"] < 0.001:
            results.append((False, "Google Ads' ROAS is almost 0: spend in the billions, because cost_micros wasn't divided by 1,000,000."))
        elif abs(roas["Google Ads"] - 2.967) < 0.001 or abs(roas["Google Ads"] - 2.5048) < 0.001:
            results.append((False, f"Google Ads' ROAS is {roas['Google Ads']:.2f}: a mean of the campaigns' or the days' ROASes. A platform's ROAS is its total revenue over its total spend."))
        elif any(abs(roas[platform] - expected) > 0.0001 for platform, (total, expected) in platforms.items()):
            results.append((False, f"roas_by_platform's roas should be each platform's revenue total over its spend total: Google Ads 2.2766, Meta 2.0194 and YouTube 1.6192. It is {roas.round(4).to_dict()}."))
        else:
            results.append((True, f"ROAS for 2024 and 2025: Google Ads {roas['Google Ads']:.2f}, Meta {roas['Meta']:.2f} and YouTube {roas['YouTube']:.2f}."))

    by_campaign = found.get("roas_by_campaign")
    if not isinstance(by_campaign, pd.DataFrame) or not {"platform", "campaign", "spend", "revenue", "roas"} <= set(by_campaign.columns):
        results.append((False, "roas_by_campaign should be a DataFrame with the columns platform, campaign, spend, revenue and roas: group with as_index=False."))
    elif len(by_campaign) != 84:
        results.append((False, f"The roas_by_campaign table has {len(by_campaign):,} rows, but there are 84 campaigns: 40 on Google Ads, 36 on Meta and 8 on YouTube."))
    elif not ((by_campaign["roas"] - by_campaign["revenue"] / by_campaign["spend"]).abs() < 1e-9).all() or abs(by_campaign["revenue"].sum() - 1216642.52) > 0.01:
        results.append((False, "The roas_by_campaign table's roas should be each campaign's revenue total over its spend total, from the campaign_days table."))
    elif by_campaign["campaign"].iloc[0] != "2024_ca_display_blackfriday" or not by_campaign["roas"].is_monotonic_decreasing:
        results.append((False, f"The roas_by_campaign table should be sorted by roas, highest first. Its first campaign is {by_campaign['campaign'].iloc[0]}."))
    else:
        results.append((True, "84 campaigns, from 2024_ca_display_blackfriday's 9.59 on $32.91 to 2025_us_facebook_blackfriday's 1.00 on $1,649.90."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run. Read your text cell again: it's what the media team will read.")


check_your_work()

## Hints

Stuck on a step? Open its hint. Each one says where the course showed it, and gives the shape of the code, with `...` where your part goes. Try the step from the hint before you open the solution.

<details>
<summary><b>Step 1: one spend table</b></summary>

Topic 5.1.: for each platform, a `rename()` dictionary from its column names to the standard ones, a `platform` column with one value, and Google Ads' micros divided by 1,000,000 (topic 4.1.). Keep the standard columns each table has, in order, and stack the three. Then compare each platform's total with its own file.

~~~python
def standard_columns(table):
    """Return the table with only the standard columns it has, in the standard order."""
    return table[[column for column in COLUMNS if column in table.columns]]


google_std = google.rename(columns={"campaign_name": "campaign"})
google_std["platform"] = "Google Ads"
google_std["spend"] = google_std["cost_micros"] / 1_000_000

meta_std = meta.rename(columns={"Reporting starts": "date", "Campaign name": ..., "Ad set name": ...,
                                "Impressions": ..., "Link clicks": ..., "Amount spent (USD)": ...})
meta_std["platform"] = ...

youtube_std = youtube.rename(columns={...})
youtube_std["platform"] = ...

spend = pd.concat([standard_columns(table) for table in [google_std, meta_std, youtube_std]], ignore_index=True)[COLUMNS]
spend.groupby("platform")["spend"].sum()
~~~

</details>

<details>
<summary><b>Step 2: revenue per date and campaign</b></summary>

Topic 5.2.: total the `sessions` table by date and `session_campaign`, so it has one row per campaign per day, like spend. Then give the key the same name as on the spend side.

~~~python
revenue = sessions.groupby([..., ...], as_index=False)[["sessions", "transactions", "revenue"]].sum()
revenue = revenue.rename(columns={...: "campaign"})
~~~

</details>

<details>
<summary><b>Step 3: campaign names that match</b></summary>

Topic 5.3.: compare the two sets of names first. Then one function that strips and lowercases, used on both tables, and the same comparison again: it should be an empty set.

~~~python
sorted(set(spend["campaign"]) - set(revenue["campaign"]))


def campaign_key(names):
    """Return campaign names stripped of spaces and lowercased, so they match across systems."""
    return names.str.strip().str.lower()


spend["campaign"] = campaign_key(spend["campaign"])
revenue["campaign"] = campaign_key(...)
assert set(spend["campaign"]) <= set(revenue["campaign"]), set(spend["campaign"]) - set(revenue["campaign"])
~~~

</details>

<details>
<summary><b>Step 4: one row per campaign-day, joined</b></summary>

Topic 5.4.: Meta has a row per ad set, so total the spend table by date, platform and campaign before you join. Then a left join from that table (topic 5.2.), with `validate="one_to_one"`, and topic 5.2.'s three checks: the rows, the spend total and the missing revenue.

~~~python
daily = spend.groupby([..., ..., ...], as_index=False)[["impressions", "clicks", "spend"]].sum()
campaign_days = daily.merge(revenue, on=[..., ...], how=..., validate=...)
assert len(campaign_days) == len(daily)
assert campaign_days["revenue"].notna().all()
assert round(campaign_days["spend"].sum(), 2) == round(spend["spend"].sum(), 2)
~~~

</details>

<details>
<summary><b>Step 5: ROAS by platform</b></summary>

Total spend and revenue per platform, and divide the totals after grouping (topic 5.2.): never a mean of the days' or the campaigns' ROASes.

~~~python
roas_by_platform = campaign_days.groupby(...)[["spend", "revenue"]].sum()
roas_by_platform["roas"] = roas_by_platform[...] / roas_by_platform[...]
~~~

</details>

<details>
<summary><b>Step 6: ROAS by campaign</b></summary>

The same, grouped by platform and campaign with `as_index=False`, so both are columns, and sorted by ROAS, highest first.

~~~python
roas_by_campaign = campaign_days.groupby([..., ...], as_index=False)[["spend", "revenue"]].sum()
roas_by_campaign["roas"] = ...
roas_by_campaign = roas_by_campaign.sort_values(..., ascending=False)
~~~

</details>

<details>
<summary><b>Step 7: the text cell</b></summary>

Choose **Insert → Text cell**. Give each platform's ROAS, and before you pick a campaign, look at its spend next to its ROAS: a high ROAS on a tiny budget says little (topic 5.2.).

</details>

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
def standard_columns(table):
    """Return the table with only the standard columns it has, in the standard order."""
    return table[[column for column in COLUMNS if column in table.columns]]


def campaign_key(names):
    """Return campaign names stripped of spaces and lowercased, so they match across systems."""
    return names.str.strip().str.lower()


google_std = google.rename(columns={"campaign_name": "campaign"})
google_std["platform"] = "Google Ads"
google_std["spend"] = google_std["cost_micros"] / 1_000_000

assert (meta["Reporting starts"] == meta["Reporting ends"]).all()
meta_std = meta.rename(columns={"Reporting starts": "date", "Campaign name": "campaign", "Ad set name": "ad_set",
                                "Impressions": "impressions", "Link clicks": "clicks", "Amount spent (USD)": "spend"})
meta_std["platform"] = "Meta"

youtube_std = youtube.rename(columns={"Day": "date", "Campaign": "campaign", "Impr.": "impressions", "Clicks": "clicks", "Cost": "spend"})
youtube_std["platform"] = "YouTube"

spend = pd.concat([standard_columns(table) for table in [google_std, meta_std, youtube_std]], ignore_index=True)[COLUMNS]
assert round(spend.loc[spend["platform"] == "Google Ads", "spend"].sum(), 2) == round(google["cost_micros"].sum() / 1_000_000, 2)
assert round(spend.loc[spend["platform"] == "Meta", "spend"].sum(), 2) == round(meta["Amount spent (USD)"].sum(), 2)
assert round(spend.loc[spend["platform"] == "YouTube", "spend"].sum(), 2) == round(youtube["Cost"].sum(), 2)
spend.groupby("platform")["spend"].sum()
```

```python
revenue = sessions.groupby(["date", "session_campaign"], as_index=False)[["sessions", "transactions", "revenue"]].sum()
revenue = revenue.rename(columns={"session_campaign": "campaign"})

spend["campaign"] = campaign_key(spend["campaign"])
revenue["campaign"] = campaign_key(revenue["campaign"])
assert set(spend["campaign"]) <= set(revenue["campaign"]), set(spend["campaign"]) - set(revenue["campaign"])
```

```python
daily = spend.groupby(["date", "platform", "campaign"], as_index=False)[["impressions", "clicks", "spend"]].sum()
campaign_days = daily.merge(revenue, on=["date", "campaign"], how="left", validate="one_to_one")
assert len(campaign_days) == len(daily)
assert campaign_days["revenue"].notna().all()
assert round(campaign_days["spend"].sum(), 2) == round(spend["spend"].sum(), 2)
len(campaign_days)
```

```python
roas_by_platform = campaign_days.groupby("platform")[["spend", "revenue"]].sum()
roas_by_platform["roas"] = roas_by_platform["revenue"] / roas_by_platform["spend"]

roas_by_campaign = campaign_days.groupby(["platform", "campaign"], as_index=False)[["spend", "revenue"]].sum()
roas_by_campaign["roas"] = roas_by_campaign["revenue"] / roas_by_campaign["spend"]
roas_by_campaign = roas_by_campaign.sort_values("roas", ascending=False)
roas_by_platform, roas_by_campaign.head(), roas_by_campaign.tail()
```

```markdown
**For the media team.** Over 2024 and 2025, Google Ads returned $2.28 of revenue for every dollar, Meta $2.02 and YouTube $1.62. Every platform made more than it spent.

The best ROAS, 2024_ca_display_blackfriday's 9.59, is on $32.91 of spend, so it says little. I'd look first at the large campaigns at the bottom: 2025_us_facebook_blackfriday made $1.00 for each of its $1,649.90, and 2025_us_display_prospecting $1.12 on $11,528.18. That's where a dollar moved would change the most.
```

</details>